#Data Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

path = "/content/drive/MyDrive/eduai-audit/data/"
df = pd.read_csv(path + "studentInfo.csv", na_values=["?"])

Mounted at /content/drive


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv(
    path + "studentInfo.csv",
    na_values=["?"]
)

df["dropout"] = (
    df["final_result"] == "Withdrawn"
).astype(int)

features = [
    "code_module",
    "code_presentation",
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "num_of_prev_attempts",
    "studied_credits",
    "disability"
]

X = df[features]
y = df["dropout"]

print("Dataset shape:", X.shape)
print("Dropout rate:", round(y.mean(), 3))

Dataset shape: (32593, 10)
Dropout rate: 0.312


#Group-Aware Data Splitting

In [ ]:
splitter1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_val_idx, test_idx = next(
    splitter1.split(
        X,
        y,
        groups=df["id_student"]
    )
)

X_train_val = X.iloc[train_val_idx]
y_train_val = y.iloc[train_val_idx]

X_test = X.iloc[test_idx]
y_test = y.iloc[test_idx]

In [ ]:
splitter2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=43
)

train_idx, val_idx = next(
    splitter2.split(
        X_train_val,
        y_train_val,
        groups=df.iloc[train_val_idx]["id_student"]
    )
)

X_train = X_train_val.iloc[train_idx]
X_val = X_train_val.iloc[val_idx]

y_train = y_train_val.iloc[train_idx]
y_val = y_train_val.iloc[val_idx]

In [ ]:
print("Train:", X_train.shape, "Dropout rate:", round(y_train.mean(), 3))
print("Validation:", X_val.shape, "Dropout rate:", round(y_val.mean(), 3))
print("Test:", X_test.shape, "Dropout rate:", round(y_test.mean(), 3))

Train: (19584, 10) Dropout rate: 0.316
Validation: (6538, 10) Dropout rate: 0.308
Test: (6471, 10) Dropout rate: 0.302


In [ ]:
train_ids = set(df.iloc[train_val_idx].iloc[train_idx]["id_student"])
val_ids = set(df.iloc[train_val_idx].iloc[val_idx]["id_student"])
test_ids = set(df.iloc[test_idx]["id_student"])

print("Train/Validation overlap:", len(train_ids & val_ids))
print("Train/Test overlap:", len(train_ids & test_ids))
print("Validation/Test overlap:", len(val_ids & test_ids))

Train/Validation overlap: 0
Train/Test overlap: 0
Validation/Test overlap: 0


In [ ]:
categorical_features = [
    "code_module",
    "code_presentation",
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability",
]

numeric_features = [
    "num_of_prev_attempts",
    "studied_credits",
]

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_processing = Pipeline([
    ("missing", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

numeric_processing = Pipeline([
    ("missing", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

preprocessing = ColumnTransformer([
    ("categorical", categorical_processing, categorical_features),
    ("numeric", numeric_processing, numeric_features),
])

In [ ]:
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("preprocessing", preprocessing),
    ("classifier", LogisticRegression(max_iter=1000)),
])

model.fit(X_train, y_train)

print("Model trained on", len(X_train), "records")

Model trained on 19584 records


In [ ]:
val_probabilities = model.predict_proba(X_val)[:, 1]

print("Validation records:", len(X_val))
print("First five predicted probabilities:", val_probabilities[:5].round(3))
print("Mean predicted probability:", round(val_probabilities.mean(), 3))
print("Observed dropout rate:", round(y_val.mean(), 3))

Validation records: 6538
First five predicted probabilities: [0.111 0.062 0.15  0.112 0.159]
Mean predicted probability: 0.314
Observed dropout rate: 0.308


In [ ]:
from sklearn.metrics import roc_auc_score

val_roc_auc = roc_auc_score(y_val, val_probabilities)
print("Validation ROC AUC:", round(val_roc_auc, 3))

Validation ROC AUC: 0.66


# Сравнение порогов на валидации

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.2, 0.3, 0.4, 0.5]
rows = []

for threshold in thresholds:
    predictions = (val_probabilities >= threshold).astype(int)
    rows.append({
        "threshold": threshold,
        "alerts": int(predictions.sum()),
        "alert_rate_pct": 100 * predictions.mean(),
        "precision": precision_score(y_val, predictions, zero_division=0),
        "recall": recall_score(y_val, predictions, zero_division=0),
        "f1": f1_score(y_val, predictions, zero_division=0),
    })

validation_results = pd.DataFrame(rows)
display(validation_results.round(3))

,threshold,alerts,alert_rate_pct,precision,recall,f1
0,0.2,5186,79.321,0.348,0.897,0.501
1,0.3,3205,49.021,0.402,0.641,0.495
2,0.4,1546,23.646,0.475,0.365,0.413
3,0.5,649,9.927,0.569,0.183,0.277


In [ ]:
comparison_rows = []
actual = y_val.to_numpy()

for threshold in [0.2, 0.3]:
    predicted = (val_probabilities >= threshold).astype(int)

    tp = int(((actual == 1) & (predicted == 1)).sum())
    fp = int(((actual == 0) & (predicted == 1)).sum())
    fn = int(((actual == 1) & (predicted == 0)).sum())
    tn = int(((actual == 0) & (predicted == 0)).sum())

    comparison_rows.append({
        "threshold": threshold,
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "true_negatives": tn,
        "false_positive_rate_pct": round(100 * fp / (fp + tn), 1),
    })

comparison = pd.DataFrame(comparison_rows)
display(comparison)

,threshold,true_positives,false_positives,false_negatives,true_negatives,false_positive_rate_pct
0,0.2,1804,3382,208,1144,74.7
1,0.3,1290,1915,722,2611,42.3


# Аудит по группам IMD

In [ ]:
def percent_or_nan(part, total):
    if total == 0:
        return np.nan
    return 100 * part / total


def audit_groups_at_threshold(y_true, probabilities, group_values, threshold):
    audit = pd.DataFrame({
        "group": pd.Series(group_values).fillna("Unknown").to_numpy(),
        "actual": np.asarray(y_true),
        "predicted": (np.asarray(probabilities) >= threshold).astype(int),
    })

    results = []
    for group_name, group in audit.groupby("group"):
        actual = group["actual"]
        predicted = group["predicted"]

        true_positives = ((actual == 1) & (predicted == 1)).sum()
        false_positives = ((actual == 0) & (predicted == 1)).sum()
        false_negatives = ((actual == 1) & (predicted == 0)).sum()
        true_negatives = ((actual == 0) & (predicted == 0)).sum()

        results.append({
            "threshold": threshold,
            "group": group_name,
            "records": len(group),
            "dropouts": int(actual.sum()),
            "alerts": int(predicted.sum()),
            "recall_pct": percent_or_nan(true_positives, true_positives + false_negatives),
            "false_positive_rate_pct": percent_or_nan(false_positives, false_positives + true_negatives),
            "alert_rate_pct": 100 * predicted.mean(),
        })

    return pd.DataFrame(results)


audit_02 = audit_groups_at_threshold(y_val, val_probabilities, X_val["imd_band"], 0.2)
audit_03 = audit_groups_at_threshold(y_val, val_probabilities, X_val["imd_band"], 0.3)
threshold_group_audit = pd.concat([audit_02, audit_03], ignore_index=True)

extreme_groups = ["0-10%", "90-100%"]
extreme_comparison = threshold_group_audit[
    threshold_group_audit["group"].isin(extreme_groups)
]

columns_to_show = [
    "threshold", "group", "records", "dropouts",
    "recall_pct", "false_positive_rate_pct",
]
display(extreme_comparison[columns_to_show].round(1).reset_index(drop=True))

,threshold,group,records,dropouts,recall_pct,false_positive_rate_pct
0,0.2,0-10%,653,236,97.5,86.8
1,0.2,90-100%,527,124,86.3,69.2
2,0.3,0-10%,653,236,83.1,60.2
3,0.3,90-100%,527,124,54.8,32.3


In [ ]:
validation_records = df.iloc[train_val_idx].iloc[val_idx]
assert validation_records.index.equals(X_val.index)
assert validation_records.index.equals(y_val.index)

bootstrap_data = pd.DataFrame({
    "student_id": validation_records["id_student"].to_numpy(),
    "group": X_val["imd_band"].fillna("Unknown").to_numpy(),
    "actual": y_val.to_numpy(),
    "probability": val_probabilities,
})

student_rows = bootstrap_data.groupby("student_id").indices
student_ids = np.array(list(student_rows))

print("Validation records:", len(bootstrap_data))
print("Unique validation students:", len(student_ids))

Validation records: 6538
Unique validation students: 5757


In [ ]:
first_group = "0-10%"
second_group = "90-100%"
thresholds_to_check = [0.2, 0.3]


def group_rate_gaps(data, threshold):
    rates = audit_groups_at_threshold(
        data["actual"], data["probability"], data["group"], threshold
    ).set_index("group")

    if first_group not in rates.index or second_group not in rates.index:
        return np.nan, np.nan

    recall_gap = rates.loc[first_group, "recall_pct"] - rates.loc[second_group, "recall_pct"]
    fpr_gap = (
        rates.loc[first_group, "false_positive_rate_pct"]
        - rates.loc[second_group, "false_positive_rate_pct"]
    )
    return recall_gap, fpr_gap


rng = np.random.default_rng(42)
n_bootstrap = 1000
bootstrap_rows = []

for repeat in range(n_bootstrap):
    drawn_students = rng.choice(student_ids, size=len(student_ids), replace=True)
    drawn_rows = np.concatenate([student_rows[student_id] for student_id in drawn_students])
    sample = bootstrap_data.iloc[drawn_rows]

    for threshold in thresholds_to_check:
        recall_gap, fpr_gap = group_rate_gaps(sample, threshold)
        bootstrap_rows.append({
            "threshold": threshold,
            "recall_gap_pp": recall_gap,
            "fpr_gap_pp": fpr_gap,
        })

bootstrap_results = pd.DataFrame(bootstrap_rows)
summary_rows = []

for threshold in thresholds_to_check:
    observed_recall, observed_fpr = group_rate_gaps(bootstrap_data, threshold)
    observed_gaps = {
        "recall_gap_pp": observed_recall,
        "fpr_gap_pp": observed_fpr,
    }

    for metric, observed_value in observed_gaps.items():
        estimates = bootstrap_results.loc[
            bootstrap_results["threshold"] == threshold, metric
        ].dropna()
        summary_rows.append({
            "threshold": threshold,
            "metric": metric,
            "observed_gap_pp": observed_value,
            "ci_low_pp": estimates.quantile(0.025),
            "ci_high_pp": estimates.quantile(0.975),
            "valid_resamples": len(estimates),
        })

bootstrap_summary = pd.DataFrame(summary_rows)
print("Group difference: 0-10% minus 90-100%")
print("Bootstrap resamples:", n_bootstrap)
display(bootstrap_summary.round(1))

Group difference: 0-10% minus 90-100%
Bootstrap resamples: 1000


,threshold,metric,observed_gap_pp,ci_low_pp,ci_high_pp,valid_resamples
0,0.2,recall_gap_pp,11.2,5.4,18.1,1000
1,0.2,fpr_gap_pp,17.6,11.8,23.3,1000
2,0.3,recall_gap_pp,28.2,18.2,38.0,1000
3,0.3,fpr_gap_pp,27.9,21.2,34.1,1000


In [ ]:
observed_02 = group_rate_gaps(bootstrap_data, 0.2)
observed_03 = group_rate_gaps(bootstrap_data, 0.3)
observed_changes = {
    "recall_gap_pp": observed_03[0] - observed_02[0],
    "fpr_gap_pp": observed_03[1] - observed_02[1],
}

change_rows = []
for metric, observed_change in observed_changes.items():
    gaps_02 = bootstrap_results.loc[
        bootstrap_results["threshold"] == 0.2, metric
    ].reset_index(drop=True)
    gaps_03 = bootstrap_results.loc[
        bootstrap_results["threshold"] == 0.3, metric
    ].reset_index(drop=True)

    changes = (gaps_03 - gaps_02).dropna()
    change_rows.append({
        "metric": metric,
        "observed_change_pp": observed_change,
        "ci_low_pp": changes.quantile(0.025),
        "ci_high_pp": changes.quantile(0.975),
        "valid_resamples": len(changes),
    })

change_summary = pd.DataFrame(change_rows)
print("Change in group gap: threshold 0.3 minus threshold 0.2")
display(change_summary.round(1))

Change in group gap: threshold 0.3 minus threshold 0.2


,metric,observed_change_pp,ci_low_pp,ci_high_pp,valid_resamples
0,recall_gap_pp,17.0,7.6,27.1,1000
1,fpr_gap_pp,10.4,4.3,16.7,1000


## Промежуточное сравнение порогов

| Порог | Ложные тревоги (FP) | Пропуски (FN) | Полнота | Разрыв полноты между крайними группами IMD | Разрыв FPR между ними |
|:---:|---:|---:|---:|---:|---:|
| 0.2 | 3 382 | 208 | 89.7% | 11.2 п. п. | 17.6 п. п. |
| 0.3 | 1 915 | 722 | 64.1% | 28.2 п. п. | 27.9 п. п. |

При пороге 0.3 ложных тревог меньше, но модель пропускает больше случаев отчисления, а различия между крайними группами IMD сильнее. Окончательный порог пока не выбран.



# Калибровка вероятностей на валидации

In [ ]:
from sklearn.metrics import brier_score_loss

calibration_data = pd.DataFrame({
    "probability": val_probabilities,
    "actual": y_val.to_numpy(),
    "imd_band": X_val["imd_band"].fillna("Unknown").to_numpy(),
})

risk_labels = ["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"]
calibration_data["risk_band"] = pd.cut(
    calibration_data["probability"],
    bins=np.linspace(0, 1, 6),
    labels=risk_labels,
    include_lowest=True,
)

overall_calibration = (
    calibration_data.groupby("risk_band", observed=True)
    .agg(
        records=("actual", "size"),
        mean_predicted_pct=("probability", lambda values: 100 * values.mean()),
        actual_dropout_pct=("actual", lambda values: 100 * values.mean()),
    )
    .reset_index()
)

print("Validation Brier score:", round(brier_score_loss(y_val, val_probabilities), 4))
display(overall_calibration.round(1))

Validation Brier score: 0.1982


,risk_band,records,mean_predicted_pct,actual_dropout_pct
0,0-20%,1352,14.5,15.4
1,20-40%,3640,29.5,29.4
2,40-60%,1335,47.7,44.9
3,60-80%,200,66.7,63.5
4,80-100%,11,83.3,72.7


In [ ]:
extreme_calibration = (
    calibration_data[calibration_data["imd_band"].isin(["0-10%", "90-100%"]) ]
    .groupby(["imd_band", "risk_band"], observed=True)
    .agg(
        records=("actual", "size"),
        mean_predicted_pct=("probability", lambda values: 100 * values.mean()),
        actual_dropout_pct=("actual", lambda values: 100 * values.mean()),
    )
    .reset_index()
)

display(extreme_calibration.round(1))

,imd_band,risk_band,records,mean_predicted_pct,actual_dropout_pct
0,0-10%,0-20%,61,14.0,9.8
1,0-10%,20-40%,339,31.3,31.3
2,0-10%,40-60%,208,48.4,44.2
3,0-10%,60-80%,42,64.7,73.8
4,0-10%,80-100%,3,84.0,33.3
5,90-100%,0-20%,141,14.6,12.1
6,90-100%,20-40%,307,28.5,23.1
7,90-100%,40-60%,73,47.7,43.8
8,90-100%,60-80%,6,69.5,66.7


В самых наполненных диапазонах прогноз и реальность в целом близки. Но для группы IMD 90-100% при прогнозе 20-40% модель дала в среднем 28,5%, а фактически отчислились 23,1% - риск может быть завышен.

In [ ]:
groups_to_compare = ["0-10%", "90-100%"]

def calibration_by_group(data):
    summary = (
        data[data["group"].isin(groups_to_compare)]
        .groupby("group")
        .agg(
            records=("actual", "size"),
            mean_predicted_pct=("probability", lambda values: 100 * values.mean()),
            actual_dropout_pct=("actual", lambda values: 100 * values.mean()),
        )
    )
    summary["overprediction_pp"] = (
        summary["mean_predicted_pct"] - summary["actual_dropout_pct"]
    )
    return summary

group_calibration = calibration_by_group(bootstrap_data)
display(group_calibration.round(1))

,records,mean_predicted_pct,actual_dropout_pct,overprediction_pp
group,,,,
0-10%,653,37.5,36.1,1.3
90-100%,527,27.9,23.5,4.4


In [ ]:
calibration_rng = np.random.default_rng(43)
calibration_resamples = 1000
calibration_bootstrap_rows = []

for _ in range(calibration_resamples):
    drawn_students = calibration_rng.choice(
        student_ids, size=len(student_ids), replace=True
    )
    drawn_rows = np.concatenate([student_rows[student_id] for student_id in drawn_students])
    sample_summary = calibration_by_group(bootstrap_data.iloc[drawn_rows])

    bias_0_10 = sample_summary.loc["0-10%", "overprediction_pp"]
    bias_90_100 = sample_summary.loc["90-100%", "overprediction_pp"]
    calibration_bootstrap_rows.append({
        "IMD 0-10%": bias_0_10,
        "IMD 90-100%": bias_90_100,
        "Difference (90-100% minus 0-10%)": bias_90_100 - bias_0_10,
    })

calibration_bootstrap = pd.DataFrame(calibration_bootstrap_rows)
observed_bias_0_10 = group_calibration.loc["0-10%", "overprediction_pp"]
observed_bias_90_100 = group_calibration.loc["90-100%", "overprediction_pp"]
observed_calibration_biases = {
    "IMD 0-10%": observed_bias_0_10,
    "IMD 90-100%": observed_bias_90_100,
    "Difference (90-100% minus 0-10%)": observed_bias_90_100 - observed_bias_0_10,
}

calibration_ci_rows = []
for metric, observed_value in observed_calibration_biases.items():
    estimates = calibration_bootstrap[metric]
    calibration_ci_rows.append({
        "metric": metric,
        "observed_pp": observed_value,
        "ci_low_pp": estimates.quantile(0.025),
        "ci_high_pp": estimates.quantile(0.975),
    })

print("Student-level bootstrap resamples:", calibration_resamples)
display(pd.DataFrame(calibration_ci_rows).round(1))

Student-level bootstrap resamples: 1000


,metric,observed_pp,ci_low_pp,ci_high_pp
0,IMD 0-10%,1.3,-2.1,5.2
1,IMD 90-100%,4.4,0.9,7.8
2,Difference (90-100% minus 0-10%),3.0,-2.0,8.0


**Промежуточный вывод по калибровке.** Для IMD 90-100% средний прогноз выше фактической доли отчислений на 4,4 п. п. (95% бутстрап-интервал от 0,9 до 7,8). Для IMD 0-10% разница составляет 1,3 п. п. (от -2,1 до 5,2). Разница между отклонениями двух групп составляет 3,0 п. п. (от -2,0 до 8,0). По этим данным пока нельзя уверенно утверждать, что общая калибровка групп различается. Это исследовательский результат на валидации.

Модель, похоже, завышает риск для группы IMD 90-100%, но пока нельзя уверенно сказать, что она завышает его сильнее, чем для IMD 0-10%.

# Сценарии проверки записей с высоким риском

In [ ]:
def top_k_scenarios(y_true, probabilities, shares, entity_ids=None):
    actual = np.asarray(y_true)
    scores = np.asarray(probabilities)

    if actual.ndim != 1 or scores.ndim != 1 or len(actual) != len(scores) or len(actual) == 0:
        raise ValueError("y_true and probabilities must be non-empty arrays of equal length")
    if not np.isin(actual, [0, 1]).all():
        raise ValueError("y_true must contain only 0 and 1")
    if not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():
        raise ValueError("probabilities must be finite numbers between 0 and 1")

    ids = None if entity_ids is None else np.asarray(entity_ids)
    if ids is not None and len(ids) != len(actual):
        raise ValueError("entity_ids must have the same length as y_true")

    ranked_positions = np.argsort(-scores, kind="stable")
    total_dropouts = int(actual.sum())
    rows = []
    flags = {}

    for share in shares:
        if not 0 < share <= 1:
            raise ValueError("Each review share must be greater than 0 and at most 1")

        records_to_review = int(np.ceil(share * len(actual)))
        selected_positions = ranked_positions[:records_to_review]
        selected = np.zeros(len(actual), dtype=bool)
        selected[selected_positions] = True
        flags[share] = selected

        found_dropouts = int(((actual == 1) & selected).sum())
        false_alarms = int(((actual == 0) & selected).sum())
        missed_dropouts = total_dropouts - found_dropouts

        row = {
            "review_share_pct": 100 * share,
            "records_to_review": records_to_review,
            "minimum_selected_risk_pct": 100 * scores[selected_positions[-1]],
            "found_dropouts": found_dropouts,
            "false_alarms": false_alarms,
            "missed_dropouts": missed_dropouts,
            "precision_pct": 100 * found_dropouts / records_to_review,
            "recall_pct": 100 * found_dropouts / total_dropouts if total_dropouts else np.nan,
        }
        if ids is not None:
            row["unique_entities_to_review"] = len(np.unique(ids[selected]))
        rows.append(row)

    return pd.DataFrame(rows), flags


review_shares = [0.10, 0.20, 0.30]
validation_actual = y_val.to_numpy()
validation_student_ids = validation_records["id_student"].to_numpy()
scenario_summary, review_flags = top_k_scenarios(
    validation_actual,
    val_probabilities,
    review_shares,
    entity_ids=validation_student_ids,
)
display(scenario_summary.round(1))

,review_share_pct,records_to_review,minimum_selected_risk_pct,found_dropouts,false_alarms,missed_dropouts,precision_pct,recall_pct,unique_entities_to_review
0,10.0,654,49.9,372,282,1640,56.9,18.5,578
1,20.0,1308,42.1,642,666,1370,49.1,31.9,1165
2,30.0,1962,37.3,888,1074,1124,45.3,44.1,1745


In [ ]:
def audit_top_k_by_group(y_true, group_values, review_flags, groups_to_show=None):
    actual = np.asarray(y_true)
    groups = pd.Series(group_values).fillna("Unknown").to_numpy()

    if len(actual) != len(groups):
        raise ValueError("y_true and group_values must have the same length")
    if groups_to_show is None:
        groups_to_show = pd.unique(groups)

    rows = []
    for share, selected in review_flags.items():
        selected = np.asarray(selected, dtype=bool)
        if len(selected) != len(actual):
            raise ValueError("Each selection mask must have the same length as y_true")

        for group_name in groups_to_show:
            in_group = groups == group_name
            if not in_group.any():
                raise ValueError(f"Group not found: {group_name}")

            group_actual = actual[in_group]
            group_selected = selected[in_group]
            total_events = int((group_actual == 1).sum())
            total_non_events = int((group_actual == 0).sum())
            found_events = int(((group_actual == 1) & group_selected).sum())
            false_alarms = int(((group_actual == 0) & group_selected).sum())

            rows.append({
                "review_share_pct": 100 * share,
                "group": group_name,
                "records": len(group_actual),
                "selected_records": int(group_selected.sum()),
                "found_events": found_events,
                "false_alarms": false_alarms,
                "recall_pct": 100 * found_events / total_events if total_events else np.nan,
                "false_positive_rate_pct": 100 * false_alarms / total_non_events if total_non_events else np.nan,
            })

    return pd.DataFrame(rows)


group_scenarios = audit_top_k_by_group(
    validation_actual,
    X_val["imd_band"],
    review_flags,
    groups_to_show=["0-10%", "90-100%"],
)
display(group_scenarios.round(1))

,review_share_pct,group,records,selected_records,found_events,false_alarms,recall_pct,false_positive_rate_pct
0,10.0,0-10%,653,122,72,50,30.5,12.0
1,10.0,90-100%,527,28,14,14,11.3,3.5
2,20.0,0-10%,653,218,109,109,46.2,26.1
3,20.0,90-100%,527,65,31,34,25.0,8.4
4,30.0,0-10%,653,318,153,165,64.8,39.6
5,30.0,90-100%,527,100,44,56,35.5,13.9


## Итог валидации

Мы обучили модель на train и проверили на validation. ROC AUC - 0,66. Модель помогает расставить записи по риску, но часто ошибается.

Мы сравнили пороги 0,2, 0,3, 0,4 и 0,5 по числу сигналов, пропусков, ложных тревог и разнице между группами. Порог 0,2 отмечал 79% записей. Порог 0,3 - 49%: ложных тревог меньше, но пропусков больше. Явного лучшего порога мы не нашли.

Чтобы заранее ограничить число проверок, для прототипа берём top-20% записей с самым высоким риском. На validation это 1 308 записей. В списке 642 реальных случая отчисления и 666 ложных сигналов. Так мы нашли 31,9% всех отчислений.

В крайних группах IMD доля найденных отчислений различается: 46,2% для 0-10% и 25,0% для 90-100%. Поэтому при дальнейшем анализе важно смотреть на результаты по группам.

Top-20% - рабочий вариант, а не доказанно лучший выбор. Test уже использовался в прошлой тетрадке при выборе порога 0,3. Для независимой финальной оценки понадобятся новые данные.